# 02 · Polynomials as Trajectories

### Recap & why now
Notebook 01 used a curve — $10\tau^3 - 15\tau^4 + 6\tau^5$ — without saying where it
came from. This notebook builds the machinery to derive it, and everything else in the
project rests on the two small functions written here.

The idea is simple enough to state in one line: a trajectory segment is a polynomial in
time, and every requirement we could have is a statement about one of its derivatives at
one instant.

### Learning objectives
1. Write a polynomial trajectory and evaluate any of its derivatives in code.
2. Count **boundary conditions** and work out the order you need.
3. Solve for the coefficients that satisfy a set of conditions.
4. Express each requirement as a **row of a matrix**, which is the pattern the whole project uses.
5. See what happens when you ask for more conditions than you have coefficients.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection used by the 3-D figures.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print numbers with 4 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=4, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Polynomial toolkit, built up over Notebooks 02-05 ===================

def poly_val(c, t, der=0):
    """Value of the polynomial c at time t, or of its `der`-th derivative."""
    out = 0.0
    for i in range(der, len(c)):                   # Terms below `der` differentiate away to zero.
        factor = 1.0
        for k in range(der):
            factor *= (i - k)                      # i(i-1)...(i-der+1), the falling factorial.
        out += c[i]*factor*t**(i - der)
    return out

def deriv_row(n, t, der):
    """Row r with r @ c = the der-th derivative at time t. One CONSTRAINT is one row."""
    r = np.zeros(n)
    for i in range(der, n):
        factor = 1.0
        for k in range(der):
            factor *= (i - k)
        r[i] = factor*t**(i - der)
    return r

def cost_matrix(n, T, der=4):
    """Q with c^T Q c = integral from 0 to T of (der-th derivative)^2 dt."""
    Q = np.zeros((n, n))
    for i in range(der, n):
        for j in range(der, n):
            ci = np.prod([i - k for k in range(der)])
            cj = np.prod([j - k for k in range(der)])
            power = i + j - 2*der + 1               # From integrating t^(i-der) * t^(j-der).
            Q[i, j] = ci*cj*T**power/power
    return Q

NCOEF = 8                                          # Order 7: eight coefficients, eight boundary conditions.
g = 9.81                                           # Gravity, needed whenever we turn acceleration into tilt.
print("polynomial toolkit ready — order %d, %d coefficients per segment per axis" % (NCOEF-1, NCOEF))

## 1 · Differentiating is a rule about exponents

$$p(t) = c_0 + c_1 t + c_2 t^2 + \dots + c_n t^n$$

$$\frac{d^{\,d}}{dt^{\,d}}\, c_i t^i = i(i-1)\cdots(i-d+1)\; c_i\, t^{\,i-d}$$

So the $d$-th derivative multiplies $c_i$ by a falling factorial and lowers the power by
$d$. That is the whole of `poly_val`, and terms below $d$ simply vanish.

In [ ]:
c_demo = np.array([1.0, -2.0, 0.5, 3.0, 0.0, -1.0])   # A polynomial of order 5.
print("  derivative   value at t = 1.7")
for d in range(4):
    print("  %10d %16.4f" % (d, poly_val(c_demo, 1.7, d)))

t = np.linspace(0, 2, 300)
fig, axes = plt.subplots(1, 4, figsize=(14, 2.7))
for d, ax_ in enumerate(axes):
    ax_.plot(t, [poly_val(c_demo, t_, d) for t_ in t], color="C0", lw=2)
    ax_.set_title(["position", "velocity", "acceleration", "jerk"][d], fontsize=10)
    ax_.set_xlabel("t")
plt.tight_layout(); plt.show()

print("Each derivative is a lower-order polynomial: the fifth-order position curve becomes a")
print("fourth-order velocity, a cubic acceleration, and a quadratic jerk. Differentiate twice")
print("more and it becomes a constant, then zero — which is what limits how many conditions")
print("a polynomial of a given order can satisfy.")

## 2 · A requirement is a row

Here is the pattern the whole project uses. Anything we could ask of a trajectory has the
same shape:

> *(some derivative) at (some time) equals (some number)*

and that is a **linear** statement about the coefficients — so it is a row of a matrix.
`deriv_row` builds it, and the entire difference between this notebook and Notebook 09 is
how many rows we stack.

In [ ]:
n = 6
print("a row is a QUESTION you can ask a trajectory:")
print("  'where does it start?'          ", deriv_row(n, 0.0, 0))
print("  'how fast does it start?'       ", deriv_row(n, 0.0, 1))
print("  'where is it at t = 2?'         ", deriv_row(n, 2.0, 0))
print("  'what is its acceleration then?'", deriv_row(n, 2.0, 2))

print("\ncheck against poly_val: %.2e" %
      abs(deriv_row(n, 1.7, 2) @ c_demo - poly_val(c_demo, 1.7, 2)))
print("\nStack the rows into A and the answers into b, and a whole specification becomes A c = b.")
print("Solving for c is then a linear system — no optimisation needed yet, and none until")
print("Notebook 05, where there are more coefficients than conditions.")

## 3 · Counting conditions

To pin down a polynomial exactly, you need as many conditions as coefficients.

* Fix **position, velocity and acceleration** at both ends: $3 \times 2 = 6$ conditions,
  so 6 coefficients, order 5.
* Add **jerk** at both ends: $4 \times 2 = 8$ conditions, order 7.

The first gives Notebook 03's minimum-jerk quintic; the second gives Notebook 04's
minimum-snap septic. Same recipe, one derivative apart.

In [ ]:
def solve_exact(conditions, n):
    """Solve for coefficients when the number of conditions equals the number of unknowns."""
    A = np.array([row for row, _ in conditions])
    b = np.array([val for _, val in conditions])
    return np.linalg.solve(A, b)                   # Square system: exactly one answer.

T = 1.0
six = [(deriv_row(6, 0.0, 0), 0.0), (deriv_row(6, 0.0, 1), 0.0), (deriv_row(6, 0.0, 2), 0.0),
       (deriv_row(6, T,   0), 1.0), (deriv_row(6, T,   1), 0.0), (deriv_row(6, T,   2), 0.0)]
c_quintic = solve_exact(six, 6)
print("six conditions, six coefficients ->", np.round(c_quintic, 4))
print("which is exactly 10t^3 - 15t^4 + 6t^5, the curve Notebook 01 used ✔\n")

eight = [(deriv_row(8, 0.0, d), 0.0) for d in range(4)] + \
        [(deriv_row(8, T, d), 1.0 if d == 0 else 0.0) for d in range(4)]
c_septic = solve_exact(eight, 8)
print("eight conditions, eight coefficients ->", np.round(c_septic, 4))
print("which is 35t^4 - 84t^5 + 70t^6 - 20t^7, the curve Notebook 04 will derive properly ✔")

## 4 · Too many conditions, and too few

Ask for more conditions than you have coefficients and the system is **overdetermined**:
usually no polynomial satisfies all of them, and NumPy will refuse.

Ask for fewer and it is **underdetermined**: many polynomials fit, and you have freedom
left over. That leftover freedom is not a nuisance — it is exactly what Notebook 05
spends on minimising a cost. The whole of trajectory optimisation lives in that gap.

In [ ]:
print("  coefficients   conditions   what happens")
for n_, k_ in [(6, 6), (8, 6), (6, 8)]:
    conds = [(deriv_row(n_, 0.0, d), 0.0) for d in range(min(4, k_//2))] + \
            [(deriv_row(n_, 1.0, d), 1.0 if d == 0 else 0.0) for d in range(min(4, k_ - k_//2))]
    A = np.array([r for r, _ in conds])
    verdict = ("exactly determined — one answer" if n_ == len(conds) else
               "%d degrees of freedom left over" % (n_ - len(conds)) if n_ > len(conds) else
               "overdetermined — usually no solution")
    print("  %12d %12d   %s" % (n_, len(conds), verdict))

A8 = np.array([r for r, _ in [(deriv_row(8, 0.0, d), 0.0) for d in range(3)] +
                              [(deriv_row(8, 1.0, d), 0.0) for d in range(3)]])
print("\nwith 8 coefficients and 6 conditions the constraint matrix has rank %d," % np.linalg.matrix_rank(A8))
print("so its null space has dimension %d — that is the room an optimiser gets to work in." %
      (8 - np.linalg.matrix_rank(A8)))
print("Notebook 05 fills that room with 'and among those, pick the smoothest'.")

## 🧪 Try it yourself

**E1.** Why do we count conditions in pairs — one at each end? What would a trajectory
with conditions only at the start look like?

**E2.** Build a quintic that starts at 0 with zero velocity, ends at 3 m with a velocity
of 1 m/s, and has zero acceleration at both ends. Plot it and check every condition.

In [ ]:
# --- Solution E1 ---
c_start_only = solve_exact([(deriv_row(6, 0.0, d), v) for d, v in
                            zip(range(6), [0.0, 0.0, 0.0, 1.0, 0.0, 0.0])], 6)
print("E1: conditions only at the start leave the END completely free — the polynomial simply")
print("    carries on. This one reaches %.2f m at t = 1 and %.2f m at t = 2, which is not a" %
      (poly_val(c_start_only, 1.0), poly_val(c_start_only, 2.0)))
print("    trajectory to a destination, it is an initial-value problem. Trajectories are")
print("    BOUNDARY-value problems: we care where they end as much as where they begin, and")
print("    that is why the conditions come in pairs.")

# --- Solution E2 ---
T = 2.0
conds = [(deriv_row(6, 0.0, 0), 0.0), (deriv_row(6, 0.0, 1), 0.0), (deriv_row(6, 0.0, 2), 0.0),
         (deriv_row(6, T,   0), 3.0), (deriv_row(6, T,   1), 1.0), (deriv_row(6, T,   2), 0.0)]
c_e2 = solve_exact(conds, 6)
print("\nE2: coefficients", np.round(c_e2, 4))
print("    condition                 asked    got")
for label, d, t_, want in [("position at 0  ", 0, 0.0, 0.0), ("velocity at 0  ", 1, 0.0, 0.0),
                           ("accel at 0     ", 2, 0.0, 0.0), ("position at T  ", 0, T, 3.0),
                           ("velocity at T  ", 1, T, 1.0), ("accel at T     ", 2, T, 0.0)]:
    print("    %s %8.3f %8.4f" % (label, want, poly_val(c_e2, t_, d)))

grid = np.linspace(0, T, 300)
fig, axes = plt.subplots(1, 3, figsize=(13, 2.7))
for d, ax_ in enumerate(axes):
    ax_.plot(grid, [poly_val(c_e2, t_, d) for t_ in grid], color="C0", lw=2)
    ax_.set_title(["position [m]", "velocity [m/s]", "accel [m/s$^2$]"][d], fontsize=10)
    ax_.set_xlabel("time [s]")
plt.tight_layout(); plt.show()
print("    Note the velocity ending at 1 m/s rather than 0 — this segment hands over to another")
print("    one still moving, which is exactly what Notebook 06 needs for flying THROUGH waypoints.")

## 🚁 Mini-project: watching the conditions bite

Animate a family of quintics, changing one boundary condition at a time, and watch the
curve reshape itself to obey. Every wiggle you see is the polynomial spending its
coefficients on the requirements you imposed.

In [ ]:
T = 2.0
end_velocities = np.concatenate([np.linspace(0, 2.5, 40), np.linspace(2.5, -2.5, 60),
                                 np.linspace(-2.5, 0, 40)])
curves = []
for v_end in end_velocities:
    conds = [(deriv_row(6, 0.0, d), 0.0) for d in range(3)] + \
            [(deriv_row(6, T, 0), 3.0), (deriv_row(6, T, 1), v_end), (deriv_row(6, T, 2), 0.0)]
    curves.append(solve_exact(conds, 6))

grid = np.linspace(0, T, 200)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.0, 3.4))

def frame(k):
    a1.clear(); a2.clear()
    cc = curves[k]
    a1.plot(grid, [poly_val(cc, t_, 0) for t_ in grid], color="C0", lw=2.2)
    a1.plot([0, T], [0, 3.0], "o", color="C3", ms=8)                 # The fixed endpoints.
    a1.set_ylim(-1, 5); a1.set_xlabel("time [s]"); a1.set_ylabel("position [m]")
    a1.set_title("end velocity = %+.2f m/s" % end_velocities[k], fontsize=10)
    a2.plot(grid, [poly_val(cc, t_, 1) for t_ in grid], color="C1", lw=2.2)
    a2.set_ylim(-4, 5); a2.set_xlabel("time [s]"); a2.set_ylabel("velocity [m/s]")
    a2.axhline(end_velocities[k], color="C3", ls=":", lw=1.4)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(curves), interval=45, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** Polynomial segments are the standard motion primitive in
> robotics and CNC machining alike, for one practical reason: they are cheap to evaluate
> and their derivatives are exact. A controller needing position, velocity and
> acceleration a thousand times a second gets all three from the same coefficients with
> no numerical differentiation — and numerical differentiation of a noisy signal is
> exactly what you do not want inside a control loop.

**Where next.** We can satisfy conditions exactly. Notebook 03 asks a better question:
among all the polynomials that satisfy them, which is the *smoothest*?